# Data Overview

This notebook explores the raw Lending Club loan dataset to understand its structure, key fields, missing values, and potential signals related to loan repayment risk.


## Raw Dataset Inspection

The raw dataset contains one or more CSV files representing historical loan records. Initial inspection focuses on understanding file size, available columns, and potential label fields without loading the entire dataset into memory.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

%matplotlib inline



In [ ]:
pd.set_option('display.max_columns', None)

DATA_RAW_PATH = "../data/raw/archive/accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv"
df = pd.read_csv(DATA_RAW_PATH, nrows=100000)

print(df)

### Dataset Size and Complexity

The accepted loans dataset contains approximately 151 columns, reflecting borrower attributes, loan terms, repayment behavior, and post-origination outcomes. Not all columns are suitable for modeling, and careful selection is required to avoid data leakage.


### Target Definition
In a loan / BNPL system, the decision is made at the time of application.

At that moment, the bank does not know the real outcome; it only has an estimate.

The actual outcome (repayment or default) is known only after time has passed, once the loan has been given.

When training a model, we learn from historical loans where the outcome is already known.

Therefore, the target variable must represent the real final outcome of past loans, because that reflects true user behavior and allows the model to learn meaningful patterns.

### Data Leakage
Any information created after the loan decision is made is not available at application time.

If such information is shown to the model during prediction, it breaks the real-world timeline.

Information like the final status of the loan directly reveals the outcome.

If the model already knows the outcome, it is no longer predicting anything.

Therefore, any information that directly or indirectly reveals the final result must never be available at prediction time.

In [ ]:
columns_list = list(df.columns)
for column in columns_list:
    print(column)

### Summary
Target:
The target represents the final outcome of a loan. In this project, the target is loan_status, which indicates whether the user eventually repaid the loan or defaulted.

Leakage:
Leakage includes any columns that should not be available to the model at the time of prediction. This includes information related to the final loan status, payment-related information, and default-related information, since these are only known after the loan has been issued.

Simple rule to avoid leakage:
Any information that would not be known at the time a user applies for a loan should not be given to the model during prediction.

### Feature Availability Filtering 

### Columns to Exclude 
- loan_status
- total_pymnt_inv
- total_rec_prncp
- total_rec_int
- total_rec_late_fee
- last_pymnt_d
- last_pymnt_amnt
- next_pymnt_d
- settlement_status
- settlement_date
- settlement_amount
- settlement_percentage
- settlement_term

This is an initial, conservative list of leakage columns and will be refined further during feature engineering and model evaluation.





In [ ]:
df.shape

In [ ]:
df.head(50)

In [ ]:

df.columns

In [ ]:
cols = list(df.columns)
cols

In [ ]:
df['loan_status'].value_counts()

In [ ]:
#filter dataset to only keep finalized loan outcomes for modeling

loan_status_list = ['Fully Paid', 'Charged Off', 'Default']
df_filtered = df[df['loan_status'].isin(loan_status_list)].copy() #create copy because its independent of the original dataframe and pandas does not get confused

df_filtered['loan_status'].value_counts()

In [ ]:
#mapping values of fully paid (0), charged off (1), default(1) in new column df_filtered[is_default]. The new column is also the target variable for
#the model. 

status_to_map = {
    'Fully Paid':0,
    'Charged Off':1,
    'Default':1
}

df_filtered['is_default'] = df_filtered['loan_status'].map(status_to_map)

df_filtered[['loan_status', 'is_default']].head(50)

### ## Feature Analysis & Data Leakage Handling

### Objective

The goal of this step is to identify which features should be used for training the model and which should be excluded to prevent data leakage and improve model reliability.

---

### Key Principle

A feature should only be used if it is available at **prediction time**.

> Any feature that contains information from the future or reflects the outcome of the loan introduces **data leakage** and must be removed.

---

### Target Variable

We defined the target variable as:

* `is_default = 1` → borrower defaulted (`Charged Off`, `Default`)
* `is_default = 0` → borrower did not default (`Fully Paid`)

---

### Feature Evaluation

#### 1. Useful Features (Kept)

* `loan_amnt` → represents total loan burden
* `term` → affects repayment duration and exposure
* `int_rate` → reflects borrower risk profile
* `installment` → represents monthly repayment burden
* `annual_inc` → indicates repayment capacity

These features are available at prediction time and are directly or indirectly related to default risk.

---

#### 2. Engineered Feature Insight

We identified that:

* `installment / annual_inc` → captures **repayment stress**

This feature provides a stronger signal by combining income and repayment burden.

---

#### 3. Dropped Features (Data Leakage)

The following features were removed because they contain **post-loan behavior**:

* `total_pymnt` → total amount paid over time
* `last_pymnt_d` → last payment date
* `next_pymnt_d` → next scheduled payment date (dynamic, reflects repayment status)

These features are not available at prediction time and indirectly reveal the loan outcome.

---

### Key Takeaways

* Features must represent information available at the time of prediction
* Avoid using features derived from future events
* Both raw and engineered features can be useful
* Understanding feature meaning is critical for building reliable models

---

### Conclusion

After this step, we have:

* A clean dataset with valid features
* No data leakage
* A well-defined target variable

This prepares the dataset for further preprocessing and modeling.


In [ ]:
cols_to_drop = ['total_pymnt', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee',
                'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d']

df_filtered = df_filtered.drop(columns=cols_to_drop) #cannot drop columns that are already dropped



In [ ]:
df_filtered.columns

### Exploratory Data Analysis 
### 'loan_amnt'

In [ ]:
df_filtered['loan_amnt'].hist()

In [ ]:
q1 = df_filtered['loan_amnt'].quantile(0.25)
q3 = df_filtered['loan_amnt'].quantile(0.75)

iqr = q3 - q1

lower_bound = q1 - (1.5 * iqr)
upper_bound =  q3 + (1.5 * iqr)

print(upper_bound)

### The upper bound is 38150.0 which tells us that the values 30k-35k are not statistical outliers and represent real-world loan amounts.
### Removing them would discard meaningful information for high-value loans. 

### EDA 
### annual_inc

In [ ]:
df_filtered['annual_inc'].hist()

### EDA for annual_inc
### The annual_inc feature is highly right-skewed.

### Most values are concentrated in the lower range, while a few very large values stretch the distribution and make it hard to visualize.

### To address this, we apply a log transformation to compress large values and better understand the distribution without removing data.

In [ ]:
df_filtered['log_annual_inc'] = np.log1p(df_filtered['annual_inc'])

In [ ]:
df_filtered['log_annual_inc'].hist()

### EDA
### instalment

In [ ]:
df_filtered['installment'].hist()

### EDA
### term

In [ ]:
df_filtered['term'].value_counts()

### EDA
### int_rate

In [ ]:
df_filtered['int_rate'].hist()

In [ ]:
missing_values = df_filtered.isnull().sum()
missing_values[missing_values > 0]

In [ ]:
df_filtered.dtypes

In [ ]:
missing_percentage = (missing_values / df_filtered.shape[0]) * 100
print(missing_percentage)

In [ ]:
high_missing = missing_percentage[missing_percentage > 70]
high_missing.shape

In [ ]:
cols_to_drop = high_missing.index

df_filtered = df_filtered.drop(columns=cols_to_drop)

In [ ]:
df_filtered.shape

In [ ]:
df_filtered['term']  = df['term']
df_filtered['term'].unique()

In [ ]:
df_filtered['term'] = df_filtered['term'].str.strip()
df_filtered['term'].unique()



In [ ]:
term_mapping = {
    '36 months' : 0,
    '60 months' : 1
}

df_filtered['term'] = df_filtered['term'].map(term_mapping)

df_filtered['term'].unique()

In [ ]:
df_filtered['grade'].unique()

In [ ]:
df_filtered['grade'] = df['grade']


In [ ]:
grade_mapping = {
    'A' : 0,
    'B' : 1,
    'C' : 2,
    'D' : 3,
    'E' : 4,
    'F' : 5,
    'G' : 6
}

df_filtered['grade'] = df_filtered['grade'].map(grade_mapping)

df_filtered['grade'].unique()

In [ ]:
df_filtered['sub_grade'].unique()

In [ ]:
sub_grades = df_filtered['sub_grade'].unique()

sorted_sub_grades = sorted(sub_grades)

sub_grade_mapping = {}

for index,value in enumerate(sorted_sub_grades):
    sub_grade_mapping[value] = index

print(sub_grade_mapping)

In [ ]:
df_filtered['sub_grade'] = df_filtered['sub_grade'].map(sub_grade_mapping)



In [ ]:
df_filtered['sub_grade'].unique()

In [ ]:
df_filtered = df_filtered.drop(columns = 'grade')

In [ ]:
df_filtered.columns

In [ ]:
df_filtered['home_ownership'].unique()

In [ ]:
home_ownership_encoded = pd.get_dummies(df_filtered['home_ownership'], drop_first=True)

home_ownership_encoded.head()

In [ ]:
df_filtered = pd.concat([df_filtered, home_ownership_encoded], axis=1)


In [ ]:
df_filtered = df_filtered.drop(columns='home_ownership')

In [ ]:
df_filtered.head()

In [ ]:
print(df_filtered['purpose'].unique())

In [ ]:
df_filtered['purpose'].nunique()

In [ ]:
purpose_encoded = pd.get_dummies(df_filtered['purpose'], drop_first=True)

purpose_encoded.head()

In [ ]:
df_filtered = pd.concat([df_filtered, purpose_encoded], axis=1)

In [ ]:
df_filtered = df_filtered.drop(columns='purpose')

In [ ]:
df_filtered.head()

In [ ]:
df_filtered.shape

In [ ]:
df_filtered.dtypes[df_filtered.dtypes == 'object']

In [ ]:
df_filtered['zip_code'].nunique()

In [ ]:
df_filtered['url'].nunique()

In [ ]:
df_filtered['emp_title'].nunique()

In [ ]:
columns_to_drop = ['emp_title', 'zip_code', 'url', 'loan_status']

df_filtered = df_filtered.drop(columns=columns_to_drop)

In [ ]:
df_filtered.shape

In [ ]:
df_filtered.dtypes[df_filtered.dtypes == 'object']

In [ ]:
df_filtered['emp_length'].unique()

In [ ]:
df_filtered['emp_length'].isnull().sum()

In [ ]:
emp_length_mapping = {
    '< 1 year' : 0,
    '1 year' : 1,
    '2 years' : 2,
    '3 years' : 3,
    '4 years' : 4,
    '5 years' : 5,
    '6 years' : 6,
    '7 years' : 7,
    '8 years' : 8,
    '9 years' : 9,
    '10+ years' : 10
}

df_filtered['emp_length'] = df_filtered['emp_length'].map(emp_length_mapping)



In [ ]:
df_filtered['emp_length'].unique()

In [ ]:
df_filtered['emp_length'].isnull().sum()

In [ ]:
df_filtered['emp_length'] = df_filtered['emp_length'].fillna(-1)

In [ ]:
df_filtered['emp_length'].isnull().sum()

In [ ]:
df_filtered['verification_status'].unique()

In [ ]:
verification_mapping = {
    'Not Verified' : 0,
    'Source Verified' : 1,
    'Verified' : 2
}

df_filtered['verification_status'] = df_filtered['verification_status'].map(verification_mapping)

In [ ]:
df_filtered['verification_status'].unique()

In [ ]:
df_filtered['pymnt_plan'].unique()

In [ ]:
df_filtered = df_filtered.drop(columns='pymnt_plan')

In [ ]:
df_filtered.shape

In [ ]:
df_filtered['initial_list_status'].unique()

In [ ]:
initial_list_status_mapping = {
    'w' : 0,
    'f' : 1
}

df_filtered['initial_list_status'] = df_filtered['initial_list_status'].map(initial_list_status_mapping)

In [ ]:
df_filtered['initial_list_status'].unique()

In [ ]:
df_filtered['application_type'].unique()

In [ ]:
app_type_mapping = {
    'Joint App' : 0,
    'Individual' : 1
}

df_filtered['application_type'] = df_filtered['application_type'].map(app_type_mapping)

In [ ]:
df_filtered['application_type'].unique()

In [ ]:
df_filtered['hardship_flag'].unique()

In [ ]:
df_filtered = df_filtered.drop(columns='hardship_flag')

In [ ]:
df_filtered.shape

In [ ]:
df_filtered['debt_settlement_flag'].unique()

In [ ]:
debt_settlement_flag_mapping = {
    'N' : 0,
    'Y' : 1
}

df_filtered['debt_settlement_flag'] = df_filtered['debt_settlement_flag'].map(debt_settlement_flag_mapping)

In [ ]:
df_filtered['debt_settlement_flag'].unique()

In [ ]:
df_filtered['disbursement_method'].unique()

In [ ]:
df_filtered = df_filtered.drop(columns='disbursement_method')

In [ ]:
df_filtered.shape

In [ ]:
df_filtered['title'].unique()

In [ ]:
df_filtered['title'].nunique()

In [ ]:
df_filtered = df_filtered.drop(columns='title')

In [ ]:
df_filtered.shape

In [ ]:
df_filtered['addr_state'].unique()

In [ ]:
df_filtered = df_filtered.drop(columns='addr_state')

In [ ]:
df_filtered.shape

In [ ]:
df_filtered['earliest_cr_line'].unique()

In [ ]:
df_filtered['earliest_cr_line'].nunique()

In [ ]:
df_filtered['earliest_cr_line'] = pd.to_datetime(df_filtered['earliest_cr_line'], format='%b-%Y')

In [ ]:
df_filtered['earliest_cr_line'].head(20)

In [ ]:
df_filtered['issue_d'].head()

In [ ]:
df_filtered['issue_d'] = pd.to_datetime(df_filtered['issue_d'], format='%b-%Y')

In [ ]:
df_filtered['issue_d'].head()

In [ ]:
df_filtered['last_credit_pull_d'].head()

In [ ]:
df_filtered['last_credit_pull_d'] = pd.to_datetime(df_filtered['last_credit_pull_d'], format='%b-%Y')

In [ ]:
df_filtered['last_credit_pull_d'].head()

In [ ]:
df_filtered.head()

In [ ]:
df_filtered.info()

In [ ]:
df_filtered.dtypes.value_counts()

In [ ]:
df_filtered.isnull().sum().sort_values(ascending=False)

In [ ]:
df_filtered.isnull().sum()[df_filtered.isnull().sum() > 0]

In [ ]:
df_filtered['mths_since_last_delinq'].dtype

In [ ]:
mths_cols = [
    'mths_since_last_delinq',
    'mths_since_recent_inq',
    'mths_since_recent_revol_delinq',
    'mths_since_recent_bc'
]

df_filtered[mths_cols] = df_filtered[mths_cols].fillna(-1)

In [ ]:
df_filtered.isnull().sum()[df_filtered.isnull().sum() > 0]

In [ ]:
credit_metric_cols = [
    'dti',
    'revol_util',
    'bc_open_to_buy',
    'bc_util',
    'mo_sin_old_il_acct',
    'num_tl_120dpd_2m',
    'percent_bc_gt_75',
    'num_rev_accts'
]

df_filtered[credit_metric_cols] = df_filtered[credit_metric_cols].fillna(df_filtered[credit_metric_cols].median())

In [ ]:
df_filtered.isnull().sum()[df_filtered.isnull().sum() > 0]

In [ ]:
df_filtered = df_filtered.dropna(subset=['last_credit_pull_d'])

In [ ]:
df_filtered.isnull().sum()[df_filtered.isnull().sum() > 0]

In [ ]:
df_filtered[['issue_d', 'last_credit_pull_d']].head(20)

In [ ]:
df_filtered = df_filtered.drop(columns=['last_credit_pull_d'])

In [ ]:
df_filtered.columns

In [ ]:
df_filtered.dtypes.value_counts()

In [ ]:
df_filtered['credit_history_age'] = (
    (df_filtered['issue_d'] - df_filtered['earliest_cr_line']).dt.days
) / 365

In [ ]:
df_filtered['credit_history_age'].head()

In [ ]:
df_filtered = df_filtered.drop(columns=['earliest_cr_line', 'issue_d'])

In [ ]:
df_filtered.dtypes.value_counts()

In [ ]:
df_filtered.columns

In [ ]:
df_filtered.shape

In [ ]:
df.shape

In [ ]:
df_filtered.head(50)

In [ ]:
status_to_map = {
    'Fully Paid': 0,
    'Charged Off': 1,
    'Default': 1
}

y = df.loc[df_filtered.index, 'loan_status'].map(status_to_map)

In [ ]:
X = df_filtered

print(X.shape)
print(y.shape)

In [ ]:
df_filtered.shape

In [ ]:
df_filtered.columns[-5:]

In [ ]:
'loan_status' in X.columns

In [ ]:
X.to_csv("../data/processed/X.csv")
y.to_csv("../data/processed/y.csv")